# Periodic primitive-cell free particle in two dimensions

**Status:** Maintained methodological review using illustrative model parameters. This is not a material-specific band-structure calculation.

**Prerequisite:** [Periodic primitive-cell free particle in two dimensions](../../../../docs/lecturenotes/solidstate/semiconductors/effmass_2d/index.md)

## Objectives

1. verify direct/reciprocal duality for a nonorthogonal primitive cell;
2. evaluate Bloch-mode kinetic energies with a Cartesian effective-mass tensor;
3. verify cell normalization and primitive-translation phases;
4. identify the contracts that must transfer to a three-dimensional primitive-cell model.


## Mathematical model

For $A=[\mathbf a_1\;\mathbf a_2]$, define

$$B=2\pi A^{-\mathsf T},\qquad \mathbf q_{\mathbf n}=\mathbf k+B\mathbf n.$$

The model evaluates

$$E_{\mathbf n}(\mathbf k)=\frac{\hbar^2}{2}\mathbf q_{\mathbf n}^{\mathsf T}M^{-1}\mathbf q_{\mathbf n}$$

and the cell-normalized plane wave

$$\psi_{\mathbf n\mathbf k}(\mathbf r)=\Omega^{-1/2}e^{i\mathbf q_{\mathbf n}\cdot\mathbf r}.$$

The laboratory uses Project Koios metal units: angstrom, dalton, picosecond, electron-volt picosecond, and electron-volt.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.solidstate.semiconductors.effmass_2d.cell import PeriodicPrimitiveCell2D
from projectkoios.physkit.solidstate.semiconductors.effmass_2d.tensor import CartesianEffectiveMassTensor2D
from projectkoios.physkit.solidstate.semiconductors.effmass_2d.model import PeriodicFreeParticle2D
from projectkoios.physkit.solidstate.semiconductors.effmass_2d.wavefunctions import PeriodicPlaneWave2DSampler, PeriodicPlaneWave2DSamplingRequest
from projectkoios.physkit.units import MODEL_SYSTEM_UNIT_CONVERTER, MatrixQuantity, PhysicalUnit, ScalarQuantity, UnitSystem, Unitless, VectorQuantity


## Nonorthogonal cell and metal-unit model

The following basis and mass tensor are illustrative synthetic inputs chosen to exercise nonorthogonality and anisotropy. They are not asserted to represent silicon or another material.


In [ ]:
basis = np.array([[5.2, 1.4], [0.0, 4.3]], dtype=np.float64)
electron_mass_dalton = MODEL_SYSTEM_UNIT_CONVERTER.convert_scalar(
    ScalarQuantity(SI.me0, PhysicalUnit("kilogram")),
    UnitSystem.METAL.mass_unit,
).magnitude
mass_tensor = electron_mass_dalton * np.array([[0.8, 0.1], [0.1, 1.3]])
cell = PeriodicPrimitiveCell2D(
    primitive_basis=MatrixQuantity(basis, UnitSystem.METAL.length_unit),
    unit_system=UnitSystem.METAL,
)
model = PeriodicFreeParticle2D(
    cell=cell,
    effective_mass=CartesianEffectiveMassTensor2D(
        tensor=MatrixQuantity(mass_tensor, UnitSystem.METAL.mass_unit),
        unit_system=UnitSystem.METAL,
    ),
)
duality = basis.T @ cell.reciprocal_basis.magnitude
assert np.allclose(duality, 2.0 * np.pi * np.eye(2), atol=1e-14)
print(f"Cell area: {cell.area.magnitude:.6f} Å²")
print("AᵀB =")
print(duality)


## Reciprocal modes at the zone center

At $\mathbf k=0$, the $(0,0)$ mode has zero energy. Opposite reciprocal indices are degenerate because the quadratic kinetic energy is even in $\mathbf q$.


In [ ]:
mode_indices = tuple((i, j) for i in range(-3, 4) for j in range(-3, 4))
gamma = VectorQuantity(np.zeros(2), PhysicalUnit("1 / angstrom"))
spectrum = model.evaluate_modes(mode_indices=mode_indices, bloch_wave_vector=gamma)
index_array = np.asarray(mode_indices)
origin = mode_indices.index((0, 0))
assert spectrum.energies.magnitude[origin] == 0.0
assert np.isclose(
    spectrum.energies.magnitude[mode_indices.index((1, -2))],
    spectrum.energies.magnitude[mode_indices.index((-1, 2))],
)
fig, ax = plt.subplots(figsize=(6.4, 5.2))
points = ax.scatter(
    spectrum.wave_vectors.magnitude[:, 0],
    spectrum.wave_vectors.magnitude[:, 1],
    c=spectrum.energies.magnitude,
    cmap="viridis",
)
for (i, j), (qx, qy) in zip(mode_indices, spectrum.wave_vectors.magnitude, strict=True):
    if abs(i) <= 1 and abs(j) <= 1:
        ax.annotate(f"({i},{j})", (qx, qy), fontsize=7)
ax.set(xlabel=r"$q_x$ (Å$^{-1}$)", ylabel=r"$q_y$ (Å$^{-1}$)", title="Reciprocal modes colored by kinetic energy")
ax.set_aspect("equal")
fig.colorbar(points, ax=ax, label="Energy (eV)")
fig.tight_layout()


## Illustrative reciprocal-coordinate path

This path reviews mode enumeration and band folding. It is not claimed to be a crystallographic high-symmetry path for the illustrative oblique cell.


In [ ]:
path_vertices = np.array([[0.0, 0.0], [0.5, 0.0], [0.5, 0.5], [0.0, 0.0]])
segments = []
for start, stop in zip(path_vertices[:-1], path_vertices[1:], strict=True):
    t = np.linspace(0.0, 1.0, 50, endpoint=False)
    segments.append(start[None, :] + t[:, None] * (stop - start)[None, :])
fractional_k = np.vstack((*segments, path_vertices[-1:]))
cartesian_k = fractional_k @ cell.reciprocal_basis.magnitude.T
small_modes = tuple((i, j) for i in range(-2, 3) for j in range(-2, 3))
energy_rows = []
for k_vector in cartesian_k:
    row = model.evaluate_modes(
        mode_indices=small_modes,
        bloch_wave_vector=VectorQuantity(k_vector, PhysicalUnit("1 / angstrom")),
    ).energies.magnitude
    energy_rows.append(np.sort(row)[:6])
energies = np.asarray(energy_rows)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(energies)
ax.set(xlabel="Path sample", ylabel="Energy (eV)", title="Six lowest folded free-particle branches")
ax.grid(alpha=0.25)
fig.tight_layout()


## Normalized Bloch plane wave

A plane wave has uniform probability density. Its real part displays the phase fronts, while integration of $|\psi|^2$ over the primitive-cell area gives one.


In [ ]:
selected = model.evaluate_modes(
    mode_indices=((1, -1),),
    bloch_wave_vector=VectorQuantity(np.array([0.08, -0.05]), PhysicalUnit("1 / angstrom")),
)
axis = np.arange(80, dtype=np.float64) / 80.0
u, v = np.meshgrid(axis, axis, indexing="ij")
fractional_points = np.column_stack((u.ravel(), v.ravel()))
sampling = PeriodicPlaneWave2DSampler().action(
    request=PeriodicPlaneWave2DSamplingRequest(
        spectrum=selected,
        fractional_coordinates=MatrixQuantity(fractional_points, Unitless()),
    )
)
psi = sampling.amplitudes.magnitude[:, 0].reshape(u.shape)
represented_norm = cell.area.magnitude * np.mean(np.abs(psi) ** 2)
assert np.isclose(represented_norm, 1.0, atol=1e-14)
x = sampling.cartesian_positions.magnitude[:, 0].reshape(u.shape)
y = sampling.cartesian_positions.magnitude[:, 1].reshape(u.shape)
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), constrained_layout=True)
phase_plot = axes[0].pcolormesh(x, y, psi.real, shading="auto", cmap="coolwarm")
axes[0].set(title=r"$\mathrm{Re}\,\psi$", xlabel="x (Å)", ylabel="y (Å)")
axes[0].set_aspect("equal")
fig.colorbar(phase_plot, ax=axes[0])
density_plot = axes[1].pcolormesh(x, y, np.abs(psi) ** 2, shading="auto", cmap="magma")
axes[1].set(title=r"$|\psi|^2$", xlabel="x (Å)", ylabel="y (Å)")
axes[1].set_aspect("equal")
fig.colorbar(density_plot, ax=axes[1], label=r"Å$^{-2}$")


## Methodological findings for the three-dimensional version

The 2D review establishes that a 3D model must retain:

- primitive vectors as columns of $A$;
- $B=2\pi A^{-\mathsf T}$ and an explicit Cartesian Bloch vector;
- integer reciprocal indices separate from physical wave vectors;
- an effective-mass tensor expressed in a declared Cartesian frame;
- explicit compound-unit conversion in the metal system;
- deterministic mode ordering and retained requests;
- volume normalization $\Omega^{-1/2}$; and
- rotation-invariance, duality, normalization, and Bloch-phase tests.

It does not yet establish a 3D implementation, a real-space finite-difference route, a DFT band structure, or material-specific effective masses.


## Exercises

1. Replace the illustrative basis with another nonorthogonal basis and verify $A^{\mathsf T}B=2\pi I$.
2. Rotate the direct basis, mass tensor, and Bloch vector together and confirm that every energy is unchanged.
3. Compare isotropic and anisotropic mass tensors without changing the reciprocal points.
4. Derive the corresponding three-dimensional volume normalization and list the additional shape checks.
5. Explain why this periodic model is not a homogeneous-Dirichlet particle in a box.
